In [8]:
!pip install pandas snowflake-connector-python

import pandas as pd
import numpy as np
import hashlib
import random
import snowflake.connector
from google.colab import userdata

ctx = snowflake.connector.connect(
    user=userdata.get('SNOWFLAKE_USER'),
    password=userdata.get('SNOWFLAKE_PASSWORD'),
    account=userdata.get('SNOWFLAKE_ACCOUNT'),
    warehouse='COMPUTE_WH',
    database='CLINICAL_SENTINEL_DB',
    schema='PUBLIC'
)

def generate_checksum(df):
    return hashlib.md5(pd.util.hash_pandas_object(df).values).hexdigest()

def generate_mock_data(n=50):
    data = []
    for i in range(1, n + 1):
        entry = {
            'LOG_ID': i,
            'FACILITY_ID': random.randint(1, 3),
            'PROVIDER_ID': random.randint(101, 103),
            'SERVICE_DATE': (pd.Timestamp.today() - pd.Timedelta(days=random.randint(0, 30))).strftime('%Y-%m-%d'),
            'SERVICE_HOURS': round(random.uniform(0.5, 8.0), 1) if random.random() > 0.1 else None,
            'HIPAA_SIGNATURE_CAPTURED': random.choice([True, True, True, False])
        }
        data.append(entry)
    return pd.DataFrame(data)

df = generate_mock_data(50)
audit_hash = generate_checksum(df)
print(f"Data Integrity Checksum: {audit_hash}")

from snowflake.connector.pandas_tools import write_pandas

try:

    success, nchunks, nrows, _ = write_pandas(ctx, df, 'FACT_SERVICE_LOGS', overwrite=True)

    print(f"Pipeline successful: {nrows} rows ingested into Snowflake.")
except Exception as e:
    print(f"Pipeline failed: {e}")
finally:
    ctx.close()

Data Integrity Checksum: f71ff3403ce5d75c99a2264c1636a3e7
Pipeline successful: 50 rows ingested into Snowflake.
